In [2]:
import os
import pandas as pd
import csv

In [4]:
def get_file_list_without_extension(directory):
    return [os.path.splitext(f)[0] for f in os.listdir(directory) 
            if os.path.isfile(os.path.join(directory, f))]

In [5]:
directory = '/Users/bz294/Documents/WhiteZone/Data/Driver_Analysis/site/'
file_list = get_file_list_without_extension(directory)
print(file_list)

['ENPLM', 'JOEBAY2E', 'ALLIGARF', 'NP-CHP', 'S197_H', 'NP-EP1', '.DS_Store', 'NP-NCL', 'NP-NMP', 'JBTS', 'OYSTERCK', 'MCCORMICK', 'TAYLORS3', 'EASTCK', 'ENPGB', 'STILLWATER', 'TROUT CR_B', 'HIGHWAY_CR', 'NP-EPSW', 'NP-P37', 'ENPTB', 'NP-OL', 'ENPLS', 'ENPTC', 'NP-146', 'SEVENPALM', 'MBTS', 'S197_T', 'SWEVER5A']


In [13]:
# Step 2: Read the CSV file
csv_path = '/Users/bz294/Documents/WhiteZone/Data/Driver_Analysis/Hydro/DBhydro/hydro_all_site_latlon.csv'
df = pd.read_csv(csv_path, header=None)  # assuming no header row

# Step 3: Filter rows where column 0 matches any name in file_names
filtered = df[df[0].isin(file_list)]

# Step 4: Extract columns 1 and 2 (second and third columns)
result = filtered[[0, 2, 3]]  # Optionally keep the name (col 0) for reference

# Example: convert to a list of tuples
result_list = list(result.itertuples(index=False, name=None))

print(result_list)



[('OYSTERCK', '251033.2', '804408.1'), ('ALLIGARF', '251032', '804734'), ('JBTS', '251328.437', '803224.22'), ('JOEBAY2E', '251357.45', '803129.52'), ('EASTCK', '251154.999', '803707.964'), ('SEVENPALM', '251028.71', '804323.2'), ('ENPGB', '251002.028', '804804.68'), ('ENPLS', '251406.576', '802724.48'), ('ENPTC', '251245.9', '803200.204'), ('SWEVER5A', '251709.99', '803421.19'), ('NP-146', '251509.072', '803958.536'), ('TAYLORS3', '251127.58', '803819.816'), ('NP-CHP', '251345.66', '804213.968'), ('NP-EPSW', '251650.124', '803029.376'), ('NP-NCL', '251439.264', '804441.352'), ('NP-NMP', '251513.86', '804753.268'), ('NP-OL', '251555.044', '803647.916'), ('NP-P37', '251709.6', '804119.356'), ('ENPLM', '251032.88', '803757.684'), ('MBTS', '251421.9', '802518.1'), ('ENPTB', '250926.424', '804329.244'), ('STILLWATER', '251343.313', '802908.621')]


In [11]:
output_path = '/Users/bz294/Documents/WhiteZone/Data/Driver_Analysis/DB_wl_latlon.csv'

# Write to CSV
with open(output_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Name', 'Lat', 'Lon'])  # optional header
    writer.writerows(result_list)

In [7]:
import pandas as pd
import os

# Directory containing the files
dir_path = "/Users/bz294/Documents/WhiteZone/Data/Driver_Analysis/Hydro/DBhydro/site"

# List of files to process
file_names = ["MBTS.csv", "S20_H.csv", "S20_T.csv",
              "MDTS.csv", "SWEVER1.csv", "S197_H.csv", "S197_T.csv"]

# List to store results
all_results = []

for file_name in file_names:
    file_path = os.path.join(dir_path, file_name)
    
    # --- Step 1: find header row ---
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if "TIMESTAMP" in line.upper():
                header_row = i
                break
        else:
            raise ValueError(f"No TIMESTAMP found in {file_name}")
    
    # --- Step 2: read CSV with detected header ---
    df = pd.read_csv(file_path, header=header_row)
    
    # Standardize column names: strip spaces, uppercase
    df.columns = [c.strip().upper() for c in df.columns]
    
    # Ensure TIMESTAMP column exists
    if "TIMESTAMP" not in df.columns:
        if "DATE" in df.columns:
            df.rename(columns={"DATE": "TIMESTAMP"}, inplace=True)
        else:
            raise ValueError(f"Cannot find TIMESTAMP column in {file_name}")
    
    # Detect VALUE column if not present
    if "VALUE" not in df.columns:
        possible_value_cols = [c for c in df.columns if "SALINITY" in c or "VALUE" in c]
        if possible_value_cols:
            df.rename(columns={possible_value_cols[0]: "VALUE"}, inplace=True)
        else:
            raise ValueError(f"Cannot find VALUE column in {file_name}")
    
    # Parse TIMESTAMP
    df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"], errors="coerce")
    
    # Convert VALUE to numeric
    df["VALUE"] = pd.to_numeric(df["VALUE"], errors="coerce")
    
    # Keep only rows with valid TIMESTAMP and VALUE
    df = df[["TIMESTAMP", "VALUE"]].dropna()
    
    # Extract year
    df["Year"] = df["TIMESTAMP"].dt.year
    
    # Compute annual mean and standard deviation
    annual_stats = df.groupby("Year")["VALUE"].agg(['mean', 'std']).reset_index()
    annual_stats.rename(columns={"mean": "mean_value", "std": "sd_value"}, inplace=True)
    
    # Add source file column
    annual_stats["source_file"] = os.path.splitext(file_name)[0]
    
    # Append to results list
    all_results.append(annual_stats)

# Combine all files into one DataFrame
combined_df = pd.concat(all_results, ignore_index=True)

# Save to single CSV
output_file = os.path.join(dir_path, "Annual_Hydro_Stats.csv")
combined_df.to_csv(output_file, index=False)

print(f"Annual statistics saved to {output_file}")
print(combined_df.head())


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_43098/2811765543.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"], errors="coerce")


Annual statistics saved to /Users/bz294/Documents/WhiteZone/Data/Driver_Analysis/Hydro/DBhydro/site/Annual_Hydro_Stats.csv
   Year  mean_value  sd_value source_file
0  1992    1.196841  1.299755        MBTS
1  1993    1.545452  0.347785        MBTS
2  1994    1.505833  0.405962        MBTS
3  1995    1.566814  0.372321        MBTS
4  1996    1.370980  0.379047        MBTS
